# Section 3 — capacity/liquidity handling starter

**Jacopo: change residual capacity handling only.** Primary destination, timing, rounding, target weights, benchmark, accounting and metric stay fixed. The 2.5% participation rate and one-day liquidation horizon are locked scenario assumptions.

[Part 2 Issue #13](https://github.com/JacopoMichelacci/Futures_Class/issues/13), Version 8 §§51–56: hard truncate (baseline), next-deferred overflow, sequential two-deferred overflow and ADV-weighted two-deferred overflow. Alternative allocation algorithms are not supplied here.

Run all cells from a fresh Python kernel using the project's existing uv environment. These notebooks use only the shared synthetic fixture and standard-library Python, with no online access. Outputs are deliberately cleared before sharing.

**Part 2 responsibility:** each teammate implements their own reusable engine and substantive tests, opens their own issue-linked PR, and reviews teammate PRs. This starter supplies inputs and interface examples, not the strand solution. Keep engine logic in `src/` and tests in `tests/unit/`. No empirical conclusions or preferred-method selection belong here. Do not consume another strand's preferred results.

See [Section 0 interfaces](../docs/section0_phase_a.md) and [fixture provenance](../data/fixtures/README.md). Phase B still needs sourced first-notice metadata, verified exchange/shortened-session times, historical benchmark/baseline quantities, production tables, the common 100-date/70% gate and AUM/capacity preflight. The synthetic calendar is not a real CBOT calendar. Daily return Tracking RMSE in basis points remains the sole ranking metric for the later historical comparison.

In [ ]:
from pathlib import Path
from decimal import Decimal
from pprint import pprint
import sys

# Start the kernel in this repository or the notebook's own folder.
cwd = Path.cwd().resolve()
ROOT = next((p for p in (cwd, *cwd.parents)
             if (p / "config/project.yaml").is_file()
             and (p / "data/fixtures/phase_a.json").is_file()
             and (p / "src/__init__.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Start the kernel in the repository or this notebook's folder.")
sys.path.insert(0, str(ROOT))
import src
assert Path(src.__file__).resolve().parent == ROOT / "src", "Restart the kernel for this checkout."
from src.validation.schemas import load_project_config, HOLDINGS_COLUMNS
from src.data.fixtures import load_fixture

config = load_project_config(ROOT / "config/project.yaml")
f = load_fixture(ROOT / "data/fixtures/phase_a.json",
                 config["max_liquidity_source_age_sessions"])
return_date = f.raw["return_dates"][0]
context = f.calendar.decision(return_date)
print("SYNTHETIC FIXTURES ONLY", config["roots"], "return interval ends", return_date)


### Publication-aware, strictly lagged ADV20

The window uses exactly 20 verified sessions ending before the decision session. Availability is checked against the actual cut-off, including revisions. Missing interior members or an endpoint outside the configured freshness limit make ADV unavailable. Do not zero-fill or replace the missing window with older observations.

In [ ]:
primary = "ZC_SYN_NEAR"  # Supplied synthetic primary, not a destination decision.
adv = f.observations.adv20(primary, context)
assert adv["available"] and len(adv["rows"]) == config["adv_lookback_days"]
assert all(r["session_index"] < context.decision_index and r["available_as_of"] for r in adv["rows"])
pprint({k: adv[k] for k in ("window_start", "window_end", "observations", "adv20_contracts")})
pprint({k: adv["rows"][-1][k] for k in ("trading_date", "publication_timestamp", "decision_cutoff")})


### Explicit metadata and supplied targets

Exact contract identity, dollar price-unit multiplier and delivery-avoidance anchor are shared inputs. The first target below is a separate hand-authored post-capacity example; its supplied ceiling was not generated by the ADV example. Keep those cases distinct and never infer a historical capacity result from either.

In [ ]:
contract = f.catalogue[primary]
pprint({**contract.identity.__dict__, "anchor": contract.anchor,
        "anchor_source": contract.anchor_source, "expiration": contract.expiration,
        "dollars_per_price_unit": contract.dollars_per_price_unit})
supplied_target = next(r for r in f.raw["post_capacity_targets"] if r["contract"] == primary)
pprint(supplied_target)


### Hand demonstration of the fixed capacity convention

This single arithmetic example illustrates the locked ceiling, not a reusable capacity engine. The engine and overflow alternatives remain your work. Overflow ordering is relative to the supplied primary contract, not absolute nearby ranks. Post-capacity fractional quantities feed the unchanged rounding stage before integer holdings reach shared accounting.

In [ ]:
from decimal import ROUND_FLOOR
adv_contracts = Decimal(adv["adv20_contracts"])
limit = int((config["capacity_participation_rate"] * adv_contracts
             * config["capacity_liquidation_horizon_days"]).to_integral_value(rounding=ROUND_FLOOR))
pprint({"decision_date": context.decision_date, "contract": primary,
        "synthetic_adv20": adv_contracts, "demonstrated_integer_ceiling": limit})
# Supplied example of the downstream shape, not output from this calculation.
pprint({key: supplied_target[key] for key in ("commodity", "contract",
        "post_capacity_fractional_contracts", "capacity_limit_contracts", "dollar_contract_notional")})


### TODO Jacopo

Implement `compute_adv20`, `integer_capacity_limit`, `apply_primary_cap`, `deferred_overflow_candidates`, `allocate_overflow_next`, `allocate_overflow_sequential`, `allocate_overflow_adv_weighted` and `build_capacity_portfolio`. Reuse Section 0's as-of ADV interface rather than redefine its timing.

Test binding/non-binding ceilings, relative deferred ordering, integer-cap compliance after rounding, sequential fills and ADV-weighted reallocation when a candidate reaches its cap. If neither candidate has valid positive ADV/capacity, leave residual unimplemented and flagged. Keep the participation scenario fixed. Produce exact-contract post-capacity fractional targets and ceilings for the unchanged rounding interface; do not implement another strand's preferred rule. Open your own issue-linked PR.

In [ ]:
# TODO: call your capacity engine, then the shared baseline rounding stage.
# Do not pass fractional implementation quantities directly to Holding.
capacity_engine_output = None
rounded_capacity_holding_rows = None
print("TODO: post-capacity targets -> unchanged rounding -> canonical integer holdings.")


### Integration boundary: future engine output → shared accounting

The TODO below is intentionally inactive. Your engine must produce canonical `Holding` rows for the baseline and all three alternatives on the same fixture dates, plus truthful decision-validity flags. Each row's `date` is the end of `(t-1,t]`; quantities and inputs are fixed by the preceding cut-off. Preserve exact contracts and explicit zero positions. Decision timestamps cannot be invented from later valuation marks.

Use the supplied fixture benchmark once and call the shared engine once for all compared methods. Missing required settlements invalidate the common date; never fill them with zero or splice maturities. One fixed AUM divides every portfolio's P&L. This adapter is for the synthetic accounting example only; the hand-authored benchmark quantities are not a constructed 60-session portfolio. Phase B must supply the actual common benchmark and frozen comparison sample.

In [ ]:
from src.portfolio.holdings import HoldingsBook
from src.metrics.portfolio_pnl import portfolio_pnl
from src.metrics.tracking_rmse_bps import tracking_rmse_bps

future_holding_rows = None  # TODO: baseline + three alternatives, produced by your engine.
future_decision_validity = None  # TODO: {(date, strategy): bool}, including the benchmark.
print("Canonical fields:", HOLDINGS_COLUMNS)
if future_holding_rows is None or future_decision_validity is None:
    print("TODO adapter: no strategy implemented or evaluated by this starter.")
else:
    benchmark_rows = [r for r in f.holdings.rows if r.strategy == f.raw["benchmark_strategy"]]
    strategies = [f.raw["benchmark_strategy"]] + sorted({r.strategy for r in future_holding_rows})
    assert len(strategies) == len(set(strategies)) == 5, "Provide baseline and three distinct alternatives."
    book = HoldingsBook(benchmark_rows + list(future_holding_rows), f.calendar, f.catalogue)
    daily = portfolio_pnl(book, f.observations, f.raw["return_dates"], strategies,
                          f.raw["fixed_aum"], f.raw["snapshot_end_exclusive"],
                          decision_validity=future_decision_validity)
    pprint(tracking_rmse_bps(daily))  # Synthetic arithmetic check; no preferred-method selection.
